# PopOut — Adversarial Search Strategies
**Artificial Intelligence 2025/2026** | DCC – FCUP

PopOut is a variant of Connect-4 where players can also remove a disc
from the bottom of a column (pop move), shifting all pieces above it
down by one row. The first player to connect four discs wins.

**Group members**
- Daniel Viloria Prieto
- Isaac Morales Santana
- Jan Henzl

---

## 1. Imports

In [2]:
from src.game.board import Board, PLAYER1, PLAYER2

from src.game.player import (
    HumanPlayer, RandomPlayer,
    MCTSPlayer, MCTSPlayerV2, MCTSPlayerV3,
    MCTSPlayerV4, MCTSPlayerV5, MCTSPlayerV6, DecisionTreePlayer
)

from src.game.game import Game
from src.decision_tree.id3 import ID3DecisionTree
from src.decision_tree.tree import DecisionTreeNode
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 2. Game Overview

The board is a 6×7 grid. Each turn a player can:
- **Drop** — place a disc into a column from the top
- **Pop** — remove their own disc from the bottom of a column

Three additional rules handle edge cases:
- If a pop creates four-in-a-row for both players, the player who popped wins
- If the board is full and a player can pop, they may declare a draw instead
- If the same state repeats three times, either player may claim a draw

## Human vs Human

In [ ]:
# Uncomment to play Human vs Human (interactive session only)
# player1 = HumanPlayer(PLAYER1)
# player2 = HumanPlayer(PLAYER2)

# game = Game(player1, player2)

# game.play()

## 4. Human vs Computer (MCTS)

The MCTS algorithm builds a search tree by iterating four phases:
1. **Selection** - descend using UCT to find a promising node
2. **Expansion** - expand one untried move
3. **Simulation** - play out a random game from that node
4. **Backpropagation** - update win/visit counts up the tree

### Why we built six variants

Pure MCTS works, but it has well-known weaknesses we wanted to address. Each variant changes **one** part of the standard algorithm so we can isolate the effect of each idea.

| Version | Simulation | Final selection | Tree reuse |
|---------|-----------|----------------|------------|
| V1 | Random | Max visits | No |
| V2 | Random | Max visits | Yes |
| V3 | Smart | Max visits | No |
| V4 | Random | Max wins | No |
| V5 | Random | Max wins | Yes |
| V6 | Smart | Max wins | No |

#### V2 - Tree reuse
**What it changes:** instead of throwing away the search tree after each move, V2 keeps the subtree rooted at the position the game actually reached.
**Why it should help:** the simulations from the previous turn already explored many of the same positions. Reusing the tree means later turns start with thousands of "free" visits already accumulated, so the same iteration budget produces stronger play as the game progresses.

#### V3 - Smart simulation
**What it changes:** during the simulation rollout (step 3), instead of picking moves purely at random, V3 always takes an immediate win when available and always blocks the opponent's immediate winning drop.
**Why it should help:** pure random rollouts often miss obvious tactical moves, so the simulated games don't reflect realistic play. A simulated opponent who never takes a winning move makes the algorithm underestimate threats. Smart simulation makes the rollouts more realistic, which makes the win/loss statistics propagated back up the tree more informative.

#### V4, V5, V6 - Max-wins final selection
**What they change:** all three replace the default "max visits" final move choice with "max wins" (which is mathematically equivalent to winrate * visits). V4 applies this on top of plain V1; V5 combines it with V2's tree reuse; V6 combines it with V3's smart simulation.
**Why it should help:** max-visits is the safe "robust child" pick but it ignores the quality of the explorations. If two children were visited equally but one has a much higher winrate, max-wins picks the better one. It rewards children that are both frequently visited and high-quality. We expect V6 to be the strongest overall, because smart simulation produces more reliable winrates per child and max-wins exploits exactly that signal - at the cost of being noticeably slower per move.


In [3]:
# player1 = HumanPlayer(PLAYER1)
# player2 = MCTSPlayerV3(PLAYER2, iterations=2000)
# 
# game = Game(player1, player2)
# 
# game.play()

=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------
INVALID INPUT: column must be a number
-------
-------
-------
-------
-------
---X---

 MCTSPlayer-O plays: ('drop', 3)
-------
-------
-------
-------
---O---
---X---
-------
-------
-------
-------
---O---
--XX---

 MCTSPlayer-O plays: ('drop', 4)
-------
-------
-------
-------
---O---
--XXO--
-------
-------
-------
-------
--XO---
--XXO--

 MCTSPlayer-O plays: ('drop', 2)
-------
-------
-------
--O----
--XO---
--XXO--
-------
-------
-------
--O----
--XO---
-XXXO--

 MCTSPlayer-O plays: ('drop', 0)
-------
-------
-------
--O----
--XO---
OXXXO--
-------
-------
-------
--OX---
--XO---
OXXXO--

 MCTSPlayer-O plays: ('drop', 1)
-------
-------
-------
--OX---
-OXO---
OXXXO--
-------
-------
---X---
--OX---
-OXO---
OXXXO--

 MCTSPlayer-O plays: ('drop', 1)
-------
-------
---X---
-OOX---
-OXO---
OXXXO--
-------
-------
---X---
-OOX---
-OXOX--
OXXXO--

 MCTSPlayer-O plays: ('drop', 1)
-------
-------
-O-X---
-

## 5. Computer vs Computer (MCTS versions)

In [ ]:
player1 = MCTSPlayer(PLAYER1, iterations=5000)
player2 = MCTSPlayerV6(PLAYER2, iterations=5000)

game = Game(player1, player2)

game.play()

=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------

 MCTSPlayerV6-X plays: ('drop', 3)
-------
-------
-------
-------
-------
---X---

 MCTSPlayer-O plays: ('drop', 3)
-------
-------
-------
-------
---O---
---X---

 MCTSPlayerV6-X plays: ('drop', 2)
-------
-------
-------
-------
---O---
--XX---

 MCTSPlayer-O plays: ('drop', 4)
-------
-------
-------
-------
---O---
--XXO--

 MCTSPlayerV6-X plays: ('drop', 1)
-------
-------
-------
-------
---O---
-XXXO--

 MCTSPlayer-O plays: ('drop', 0)
-------
-------
-------
-------
---O---
OXXXO--

 MCTSPlayerV6-X plays: ('drop', 2)
-------
-------
-------
-------
--XO---
OXXXO--

 MCTSPlayer-O plays: ('drop', 2)
-------
-------
-------
--O----
--XO---
OXXXO--

 MCTSPlayerV6-X plays: ('drop', 4)
-------
-------
-------
--O----
--XOX--
OXXXO--

 MCTSPlayer-O plays: ('drop', 1)
-------
-------
-------
--O----
-OXOX--
OXXXO--

 MCTSPlayerV6-X plays: ('drop', 4)
-------
-------
-------
--O-X--
-OXOX--
OXXXO--

 MCTSPlayer

## 7. Decision Tree (ID3)
The ID3 procedure builds a decision tree by recursively selecting the attribute that maximises information gain (entropy reduction).

For continuous attributes we find the optimal split threshold automatically.

### 7.1 Dataset 1 - Iris (warm-up)


In [ ]:
# Load the iris dataset
from src.game.dataset_generator import load_iris
X_iris, y_iris, iris_features = load_iris('data/iris.csv')

print(f'Iris dataset: {len(X_iris)} samples, {len(iris_features)} features')
print('Features:', iris_features)
print('Classes:', sorted(set(y_iris)))

df = pd.DataFrame(X_iris, columns=iris_features)
df['class'] = y_iris
df.describe()

In [ ]:
# Train / test split (80 / 20)
import random
random.seed(42)
indices = list(range(len(X_iris)))
random.shuffle(indices)
split = int(0.8 * len(indices))

X_train = [X_iris[i] for i in indices[:split]]
y_train = [y_iris[i] for i in indices[:split]]
X_test  = [X_iris[i] for i in indices[split:]]
y_test  = [y_iris[i] for i in indices[split:]]

print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')

In [ ]:
# Build the ID3 decision tree (all four features are continuous)
iris_tree = ID3DecisionTree(
    continuous_features=list(range(4)),
    feature_names=iris_features,
)
iris_tree.fit(X_train, y_train)

print('=== Iris Decision Tree ===')
iris_tree.display()

train_eval = iris_tree.evaluate(X_train, y_train)
test_eval  = iris_tree.evaluate(X_test,  y_test)
print(f'\nTrain accuracy: {train_eval["accuracy"]:.2%}  ({train_eval["correct"]}/{train_eval["total"]})')
print(f'Test  accuracy: {test_eval["accuracy"]:.2%}  ({test_eval["correct"]}/{test_eval["total"]})')

In [ ]:
# Effect of max_depth on Iris accuracy
depths = [1, 2, 3, 4, 5, None]
train_accs, test_accs = [], []

for d in depths:
    t = ID3DecisionTree(continuous_features=list(range(4)), max_depth=d)
    t.fit(X_train, y_train)
    train_accs.append(t.evaluate(X_train, y_train)['accuracy'])
    test_accs.append(t.evaluate(X_test,   y_test)['accuracy'])

depth_labels = [str(d) if d is not None else 'None' for d in depths]
x = range(len(depths))

plt.figure(figsize=(8, 4))
plt.plot(x, train_accs, marker='o', label='Train')
plt.plot(x, test_accs,  marker='s', label='Test')
plt.xticks(x, depth_labels)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Iris — Effect of max_depth on ID3 accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iris_depth.png', dpi=80)
plt.show()
plt.close()
print('Plot saved to iris_depth.png')

### 7.2 Dataset 2 - PopOut Game
We generate a labelled dataset using MCTS self-play. Each sample is a board state (42 cells + 1 current player = 43 features). The label is the MCTS-recommended move as movetype_col (e.g. d3).

In [ ]:
from src.game.dataset_generator import generate_dataset

generate_dataset(800, "data/d4.csv", 10000)
df = pd.read_csv("data/d4.csv")

print(f"Total samples: {len(df)}")
print(f"Drop moves: {(df['move_type'] == 'drop').sum()}")
print(f"Pop moves:  {(df['move_type'] == 'pop').sum()}")

df.head()

In [ ]:
# Load the popout dataset
from src.game.dataset_generator import load_popout
X_popout, y_popout, popout_features = load_popout('data/popout.csv')

print(f'PopOut dataset: {len(X_popout)} samples, {len(popout_features)} features')
print('Features:', popout_features)
print('Classes:', sorted(set(y_popout)))

df = pd.DataFrame(X_popout, columns=popout_features)
df['class'] = y_popout
df.describe()

In [ ]:
# Train / test split for PopOut dataset
import random
random.seed(0)
idx = list(range(len(X_popout)))
random.shuffle(idx)
sp = int(0.8 * len(idx))

Xp_train = [X_popout[i] for i in idx[:sp]]
yp_train = [y_popout[i] for i in idx[:sp]]
Xp_test  = [X_popout[i] for i in idx[sp:]]
yp_test  = [y_popout[i] for i in idx[sp:]]

print(f'Train: {len(Xp_train)} | Test: {len(Xp_test)}')

In [ ]:
# Train the ID3 decision tree on the PopOut dataset
# Board cells are categorical (values: 0, 1, 2) → no continuous_features needed
popout_tree = ID3DecisionTree(
    continuous_features=[],
    feature_names=popout_features,
    max_depth=50,
)
popout_tree.fit(Xp_train, yp_train)

pte = popout_tree.evaluate(Xp_train, yp_train)
ppe = popout_tree.evaluate(Xp_test,  yp_test)

print(f'Train accuracy: {pte["accuracy"]:.2%}')
print(f'Test  accuracy: {ppe["accuracy"]:.2%}')


In [ ]:
# simple grid search with your ID3 class
depths = [1,2,3,4,5,10,None]
best = (None, -1)
for d in depths:
    t = ID3DecisionTree(continuous_features=[], feature_names=popout_features, max_depth=d)
    t.fit(Xp_train, yp_train)
    acc = t.evaluate(Xp_test, yp_test)['accuracy']
    print('max_depth=', d, 'test_acc=', acc)
    if acc > best[1]:
        best = (d, acc)
print('best', best)

In [ ]:
def run_tournament(player1, player2, n_games: int = 5, verbose: bool = False):
    """Run n_games between player1 and player2 and return win counts."""
    results = {"player1": 0, "player2": 0, "draw": 0}
    for _ in range(n_games):
        game = Game(player1, player2)
        result = game.play()
        if result == PLAYER1:
            results["player1"] += 1
        elif result == PLAYER2:
            results["player2"] += 1
        else:
            results["draw"] += 1
    return results


dt_player   = DecisionTreePlayer(PLAYER1, popout_tree)
mcts_player = MCTSPlayer(PLAYER2, 500)

print('Decision Tree player vs. MCTS player (50 iters) — 5 games')
results = run_tournament(dt_player, mcts_player, n_games=10)
print(f'  Decision Tree wins: {results["player1"]}')
print(f'  MCTS wins:          {results["player2"]}')
print(f'  Draws:              {results["draw"]}')